In [115]:
# import statements
import pandas as pd
import requests
import json

import matplotlib.pyplot as plt


## Data Handling + Split

In [116]:
#call data and clean (remove null and duplicate values)
df = pd.read_csv('agriculture_dataset.csv')
df=df.drop_duplicates()

missing_codes = ["--", "", " ", "nan", "NaN", "None"] #account for different possible null combos


for col in df.columns:
    if df[col].dtype == object:  # only clean string/object columns
        df[col] = df[col].str.strip().replace(missing_codes, pd.NA)
    else:
        df[col] = df[col].replace(missing_codes, pd.NA) 

print(df.columns)
df.head()

Index(['High_Resolution_RGB', 'Multispectral_Images', 'Thermal_Images',
       'Temporal_Images', 'Spatial_Resolution', 'GPS_Coordinates',
       'Field_Boundaries', 'Elevation_Data', 'Canopy_Coverage', 'NDVI', 'SAVI',
       'Chlorophyll_Content', 'Leaf_Area_Index', 'Crop_Stress_Indicator',
       'Temperature', 'Humidity', 'Rainfall', 'Wind_Speed', 'Soil_Moisture',
       'Soil_pH', 'Organic_Matter', 'Pest_Hotspots', 'Weed_Coverage',
       'Pest_Damage', 'Crop_Growth_Stage', 'Expected_Yield', 'Crop_Type',
       'Ground_Truth_Segmentation', 'Bounding_Boxes', 'Water_Flow',
       'Drainage_Features', 'Crop_Health_Label'],
      dtype='object')


,High_Resolution_RGB,Multispectral_Images,Thermal_Images,Temporal_Images,Spatial_Resolution,GPS_Coordinates,Field_Boundaries,Elevation_Data,Canopy_Coverage,NDVI,...,Weed_Coverage,Pest_Damage,Crop_Growth_Stage,Expected_Yield,Crop_Type,Ground_Truth_Segmentation,Bounding_Boxes,Water_Flow,Drainage_Features,Crop_Health_Label
0,0,0,0,0,0.667324,201538,3,28.207634,8.046926,0.676945,...,1.922274,84,2,2540.784327,Wheat,1,5,41.771884,0,1
1,1,1,0,0,1.459000,215854,3,82.335147,147.512332,0.414781,...,4.851381,56,3,3227.617025,Wheat,0,1,27.564635,0,1
2,0,0,0,0,0.500442,890802,3,83.865629,30.246527,0.723610,...,5.974859,38,1,4609.938146,Maize,1,8,29.510836,0,1
3,0,0,0,0,1.865161,605584,3,20.747905,6.857820,0.405611,...,2.100598,27,2,1409.716754,Maize,0,1,34.822855,0,0
4,0,1,1,1,1.392331,871732,3,22.588815,26.168558,0.465992,...,3.025669,84,4,3905.312588,Rice,0,2,15.493255,1,0


In [117]:
#create df_sub and df_sub_drop (clean dataset/create subsets)
df_sub = df.drop(columns='Crop_Health_Label', axis=1)
df_sub = pd.get_dummies(df_sub, columns=['Crop_Type'], dtype=int)

columnsToDrop = ['High_Resolution_RGB','Multispectral_Images', 'Thermal_Images',
       'Temporal_Images','Field_Boundaries', 'Pest_Hotspots','Crop_Growth_Stage',
       'Crop_Type_Maize','Crop_Type_Rice','Crop_Type_Wheat', 
       'Ground_Truth_Segmentation', 'Bounding_Boxes', 'Drainage_Features']
df_sub_drop = df_sub.drop(columns=columnsToDrop, axis=1)

df_sub_drop.head()


,Spatial_Resolution,GPS_Coordinates,Elevation_Data,Canopy_Coverage,NDVI,SAVI,Chlorophyll_Content,Leaf_Area_Index,Crop_Stress_Indicator,Temperature,Humidity,Rainfall,Wind_Speed,Soil_Moisture,Soil_pH,Organic_Matter,Weed_Coverage,Pest_Damage,Expected_Yield,Water_Flow
0,0.667324,201538,28.207634,8.046926,0.676945,0.475536,0.829063,3.107188,79,24.627325,47.240283,7.056089,2.795500,31.010549,6.085810,2.126335,1.922274,84,2540.784327,41.771884
1,1.459000,215854,82.335147,147.512332,0.414781,0.325712,0.435861,1.287952,88,27.671999,44.408156,14.005230,2.325063,12.429003,6.776880,1.751158,4.851381,56,3227.617025,27.564635
2,0.500442,890802,83.865629,30.246527,0.723610,0.511144,1.001452,1.229495,59,23.515820,56.268020,46.152684,1.705493,28.454713,6.078783,0.365124,5.974859,38,4609.938146,29.510836
3,1.865161,605584,20.747905,6.857820,0.405611,0.162857,0.962720,0.995907,11,11.473797,38.637011,15.168736,5.891225,28.450994,6.596315,2.484465,2.100598,27,1409.716754,34.822855
4,1.392331,871732,22.588815,26.168558,0.465992,0.269888,2.111205,0.438672,79,22.502605,64.438963,9.492281,2.281165,8.295005,7.664008,3.520753,3.025669,84,3905.312588,15.493255


In [118]:
#binary columns to add back
binary_cols = df_sub[['Pest_Hotspots','Crop_Growth_Stage',
       'Crop_Type_Maize','Crop_Type_Rice','Crop_Type_Wheat', 
       'Ground_Truth_Segmentation', 'Bounding_Boxes', 'Drainage_Features']].to_numpy()


In [119]:
from sklearn.model_selection import train_test_split
#split continuous and binary data at same time, consistent rand state
y_vec = df['Crop_Health_Label'].to_numpy()

X_train_cont, X_test_cont, y_train, y_test = train_test_split(
    df_sub_drop.to_numpy(), y_vec, test_size=0.2, random_state=3000
)
binary_train, binary_test, _, _ = train_test_split(
    binary_cols, y_vec, test_size=0.2, random_state=3000
)

In [120]:
import numpy as np

#scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cont)
X_test_scaled = scaler.transform(X_test_cont)

#add binary columns back
X_train = np.concatenate([X_train_scaled, binary_train], axis=1)
X_test = np.concatenate([X_test_scaled, binary_test], axis=1)

pd.DataFrame(X_train_scaled).head()
#pd.DataFrame(X_test_scaled).head()



,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1.258298,-1.124710,0.710358,-0.961950,1.170413,0.315590,-0.391386,0.826635,1.157794,0.582669,0.351784,-0.146676,0.140998,0.695356,-2.308856,-0.238568,-0.710386,0.989493,-1.308874,-0.110975
1,0.860816,0.792532,0.993023,-0.836669,-0.350865,0.011197,-0.172440,-0.706246,-0.884847,-1.092459,1.497361,-0.186553,1.709032,0.184990,1.077563,-0.990301,0.434284,1.578311,0.286524,-0.989180
2,0.603661,1.611461,1.339824,0.506662,0.677569,0.883997,-0.362928,-0.966201,-1.231057,-1.376744,0.844621,0.501672,1.151713,1.603062,-1.543268,-0.861316,0.551110,-1.157961,-0.313284,-0.931432
3,0.410018,-0.045530,0.074220,-0.081565,-0.515775,0.516771,-1.066198,-0.217016,-0.157805,0.195901,-1.695634,-0.109149,0.219839,-1.167445,-0.328454,-0.810994,-0.149662,-0.534506,0.253110,-0.192515
4,-0.994252,-1.500975,0.318728,4.638078,-0.134009,0.574754,1.286929,-0.333555,-1.438784,0.285036,1.000690,-0.207189,-0.295081,1.685137,0.184746,-0.676146,-1.113010,-0.014961,0.623033,1.002729


## KNN Regressor

In [121]:
#Cosine KNN-model1
def knn_reg_cos(X_train, y_train, X_test, k):
    """
    Args:
    X_train: numpy array of shape (n_samples, n_features) - training data features
    y_train: numpy array of shape (n_samples,) - training data target values
    X_test: numpy array of shape (m_samples, n_features) - test data features
    k: int - number of nearest neighbors to consider

    Returns:
    y_pred: numpy array of shape (m_samples,) - predicted target values for the test data
    nearest_neighbors: list of lists - indices of the k nearest neighbors for each test point
    """

    #get norms
    train_norms = np.linalg.norm(X_train, axis=1, keepdims=True)
    test_norms = np.linalg.norm(X_test, axis=1, keepdims=True)
    
    X_train_norm = X_train / train_norms
    X_test_norm = X_test / test_norms
    
    # All cosine sims in one matrix multiply
    similarity_matrix = X_test_norm @ X_train_norm.T
    
    neighbor_indices = np.argsort(similarity_matrix, axis=1)[:, -k:]
    y_pred = y_train[neighbor_indices].mean(axis=1)
    
    return y_pred, neighbor_indices.tolist()

In [122]:
#l2 norm-model 2
def knn_reg_l2(X_train, y_train, X_test, k):
    """
    Args:
    X_train: numpy array of shape (n_samples, n_features) - training data features
    y_train: numpy array of shape (n_samples,) - training data target values
    X_test: numpy array of shape (m_samples, n_features) - test data features
    k: int - number of nearest neighbors to consider
    
    Returns:
    y_pred: numpy array of shape (m_samples,) - predicted target values for the test data
    nearest_neighbors: list of lists - indices of the k nearest neighbors for each test point
    """
    train_sq = np.sum(X_train**2, axis=1)
    test_sq = np.sum(X_test**2, axis=1)
    
    asqr=test_sq[:, None]
    bsqr=train_sq[None, :]

    ab=X_test @ X_train.T


    dist_matrix = asqr + bsqr - 2 * (ab)
    dist_matrix = np.sqrt(np.maximum(dist_matrix, 0))  # avoid negative sqrt
    
    neighbor_indices = np.argsort(dist_matrix, axis=1)[:, :k]
    y_pred = y_train[neighbor_indices].mean(axis=1)
    
    return y_pred, neighbor_indices.tolist()

## plot data with ks and model

In [123]:
# loop through different values of k, keeping track of MSE for the test set predictions
# first with cosine similarity
k_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mse_values = []
for k in k_values:
    y_pred, _ = knn_reg_cos(X_train, y_train, X_test, k)
    mse = np.mean((y_test - y_pred) ** 2)
    mse_values.append(mse)

# plot the MSE values for different k
plt.figure(figsize=(10, 6))
plt.plot(k_values, mse_values, marker='o')
plt.title('MSE of k-NN Regressor with Cosine Similarity')
plt.xlabel('k (number of neighbors)')
plt.ylabel('Mean Squared Error')
plt.xticks(k_values)
plt.grid()

MemoryError: Unable to allocate 53.6 GiB for an array with shape (42404, 169615) and data type float64

In [ ]:
#l2 norm model
mse_values_l2 = []
for k in k_values:
    y_pred_l2, _ = knn_reg_l2(X_train, y_train, X_test, k)
    mse_l2 = np.mean((y_test - y_pred_l2) ** 2)
    mse_values_l2.append(mse_l2)

# plot the MSE values for different k for L2 distance
plt.figure(figsize=(10, 6))
plt.plot(k_values, mse_values_l2, marker='o', color='orange')
plt.title('MSE of k-NN Regressor with L2 Distance')
plt.xlabel('k (number of neighbors)')
plt.ylabel('Mean Squared Error')
plt.xticks(k_values)
plt.grid()

MemoryError: Unable to allocate 53.6 GiB for an array with shape (42404, 169615) and data type float64